# 01 — PCA + kNN baseline

**What this does:** fixes the donor-held-out split (or loads the existing
one), then builds a deliberately simple classifier on the *same 2,000
reference-derived HVGs* the deep models use:

`counts → normalize_total → log1p → PCA (fit on reference) → kNN (fit on
reference labels) → predict query`.

**Leakage controls:** all learned transforms (HVG list, PCA, kNN) are fit on
reference cells only; the query object contains no cell-type labels and this
notebook never opens `query_eval_labels_main.csv`.

**Expected output:** `results/predictions/baseline_predictions_main.csv`.

In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
if (REPO / "src").exists():
    sys.path.insert(0, str(REPO / "src"))
import heartmap
print("heartmap from:", Path(heartmap.__file__).parent)


In [ ]:
from heartmap.config import load_config
cfg = load_config("configs/main.yaml")
print("reference:", cfg.reference_path)
print("query    :", cfg.query_model_input_path)
print("HVG list :", cfg.hvg_path)


## 1. (Re)create the sealed split if needed

Safe to skip when the split files already exist. The query donor is D6, fixed deterministically before any training.

In [ ]:
from pathlib import Path
from heartmap.data import load_heart_dataset, validate_counts
from heartmap.split import make_split

if not cfg.reference_path.exists():
    adata = load_heart_dataset(str(cfg.data_raw_dir),
                               remove_nuisance_clusters=True)
    adata.layers[cfg["counts_layer"]] = adata.X.copy()
    validate_counts(adata, cfg["counts_layer"])
    make_split(adata, cfg)
    print("split created")
else:
    print("split already fixed; nothing to do")


## 2. Load reference and sealed query

In [ ]:
import anndata as ad
reference = ad.read_h5ad(cfg.reference_path)
query = ad.read_h5ad(cfg.query_model_input_path)
print(reference.shape, query.shape)
assert "cell_type" not in query.obs.columns
assert set(query.obs["labels_scanvi"].astype(str)) == {"Unknown"}


## 3. Fit and run the baseline

`run_baseline` loads the frozen reference HVG list, fits PCA and kNN on reference cells only, transforms the query, and returns predictions plus fit metadata (PCA variance, k, timings). The HVG matrix is densified once for sklearn PCA (~0.125 GB at main size).

In [ ]:
from heartmap.baseline import run_baseline
preds, meta = run_baseline(reference, query, cfg)
preds.head()


## 4. Inspect fit metadata

Note `query_true_labels_read: False` — the baseline never accesses ground truth.

In [ ]:
{ k: v for k, v in meta.items() if not k.endswith("_seconds") and
  k != "pca_explained_variance_ratio" }


In [ ]:
import numpy as np
evr = np.asarray(meta["pca_explained_variance_ratio"])
print("PCA cumulative variance (30 comps):", float(evr.sum().round(4)))


## 5. Confidence distribution (labels still sealed)

Confidence is the distance-weighted neighbour vote fraction of the winning class — a score, not a calibrated probability.

In [ ]:
import matplotlib.pyplot as plt
ax = preds["confidence"].hist(bins=40, figsize=(6, 3))
ax.set_xlabel("kNN vote confidence"); ax.set_ylabel("cells")
plt.show()
print(preds["predicted_label"].value_counts())


## 6. Save frozen predictions

No true-label column is written. Evaluation happens only in notebook 04, after both prediction sets are frozen.

In [ ]:
out = cfg.results_dir / "predictions" / f"baseline_predictions_{cfg.run_tag}.csv"
out.parent.mkdir(parents=True, exist_ok=True)
preds.to_csv(out, index=False)
print("wrote", out)
